In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import os
from openai import OpenAI
from dotenv import load_dotenv
import warnings

warnings.filterwarnings('ignore')
load_dotenv()

# ============================================================================
# Project paths
# ============================================================================
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODEL_DIR = RESULTS_DIR / 'model'
TABLE_DIR = RESULTS_DIR / 'table'
REPORTS_DIR = RESULTS_DIR / 'reports'

for d in [DATA_DIR, MODEL_DIR, TABLE_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project paths ready")

print("=" * 80)
print("Step 6: Agent #2 — Consulting Report Generator (Full Run)")
print("=" * 80)

# ============================================================================
# 1. Feature Name Mapping — reused from Agent #1 (Step 5) for consistency.
#    Base term dictionary + suffix-aware readable-name function, so ratio
#    features (e.g. 'FN1_16_to_assets') resolve correctly instead of falling
#    back to the raw code.
# ============================================================================
BASE_TERM_MAPPING = {
    'FN1_1': 'Current Assets', 'FN1_2': 'Non-Current Assets', 'FN1_3': 'Quick Assets',
    'FN1_4': 'Inventory', 'FN1_5': 'Tangible Assets', 'FN1_6': 'Work in Process',
    'FN1_7': 'Cash', 'FN1_8': 'Cash Equivalents', 'FN1_9': 'Marketable Securities',
    'FN1_10': 'Cash and Cash Equivalents', 'FN1_11': 'Accounts Receivable',
    'FN1_11_2': 'Loss on Disposal of Receivables', 'FN1_11_3': 'Intangible Assets',
    'FN1_11_4': 'Investment Assets', 'FN1_14': 'Current Liabilities',
    'FN1_15': 'Short-Term Borrowings', 'FN1_16': 'Borrowings', 'FN1_17': 'Accounts Payable',
    'FN1_18': 'Non-Current Liabilities', 'FN1_19': 'Total Liabilities',
    'FN1_20': 'Paid-in Capital', 'FN1_21': 'Capital Surplus', 'FN1_21_1': 'Paid-in Capital (Detail)',
    'FN1_22': 'Retained Earnings', 'FN1_22_1': 'Capital Adjustments',
    'FN1_22_2': 'Accumulated Other Comprehensive Income', 'FN1_23': 'Reserves',
    'FN1_24': 'Total Equity', 'FN3_10_1': 'Liquidation Value', 'FN3_11': 'Net Working Capital',
    'FN3_11_1': 'Net Borrowings',
    'FN2_2': 'Cost of Goods Sold', 'FN2_2_1': 'Gross Profit', 'FN2_3': 'SG&A Expenses',
    'FN2_3_1': 'Pre-Tax Income', 'FN2_3_2': 'Prior-Year Pre-Tax Income', 'FN2_3_3': 'Corporate Tax',
    'FN2_3_4': 'Income from Continuing Operations', 'FN2_3_5': 'Discontinued Operations Gain/Loss',
    'FN2_4': 'Financial Expenses', 'FN2_5': 'Operating Income', 'FN2_5_1': 'Prior-Year Operating Income',
    'FN2_7': 'Non-Operating Income', 'FN2_8': 'Non-Operating Expenses', 'FN2_9': 'Pre-Tax Net Income',
    'FN2_10': 'Net Income', 'FN3_1': 'Cash Flow', 'FN3_2': 'Operating Cash Flow',
    'FN3_2_1': 'Investing Cash Flow', 'FN3_2_2': 'Financing Cash Flow', 'FN3_4_1': 'Interest Expense',
    'FN3_4_2': 'Bond Interest', 'FN3_7': 'EBIT', 'FN3_8': 'EBITDA',
    'FN3_3': 'Debt Service Coverage Ratio', 'FN3_6': 'Reserve Ratio', 'FN3_10': 'Liquidation Value Ratio',
    'asset_growth_rate': 'Total Asset Growth Rate', 'revenue_growth_rate': 'Revenue Growth Rate',
    'operating_income_growth': 'Operating Income Growth Rate', 'net_income_growth': 'Net Income Growth Rate',
    'equity_growth_rate': 'Equity Growth Rate',
}

# Categories used for directional-hint logic — keyed on the BASE term, so a
# ratio feature like 'FN1_16_to_assets' resolves its base ('FN1_16' ->
# 'Borrowings') to the 'liability_like' category, independent of whether the
# suffix is '_to_assets' or '_to_revenue'. This fixes the earlier bug where
# a string-contains check on the full readable name (e.g. "Borrowings (% of
# Total Assets)") matched "Assets" and misclassified a liability variable as
# an asset-expansion signal.
LIABILITY_LIKE_BASES = {'FN1_15', 'FN1_16', 'FN1_18', 'FN1_19', 'FN3_11_1'}
ASSET_LIKE_BASES = {'FN1_1', 'FN1_2', 'FN1_3', 'FN1_4', 'FN1_5', 'FN1_9', 'FN1_11_3', 'FN1_11_4'}
RECEIVABLE_BASES = {'FN1_11'}

def get_base_feature(feature_name):
    """Strips the ratio/growth suffix to recover the original account code."""
    for suffix in ['_to_assets', '_to_revenue']:
        if feature_name.endswith(suffix):
            return feature_name[:-len(suffix)]
    return feature_name  # growth-rate features and Group-A ratios have no suffix to strip

def get_readable_name(feature_name):
    if feature_name in BASE_TERM_MAPPING:
        return BASE_TERM_MAPPING[feature_name]
    if feature_name.endswith('_to_assets'):
        base = feature_name[:-len('_to_assets')]
        base_term = BASE_TERM_MAPPING.get(base, base)
        return f"{base_term} (% of Total Assets)"
    if feature_name.endswith('_to_revenue'):
        base = feature_name[:-len('_to_revenue')]
        base_term = BASE_TERM_MAPPING.get(base, base)
        return f"{base_term} (% of Revenue)"
    return feature_name

# ============================================================================
# 2. Industry Context — same 4-sector personas as Agent #1, for consistency
#    across the pipeline.
# ============================================================================
INDUSTRY_CONTEXT = {
    'G46': {'name': 'wholesale trade'},
    'G47': {'name': 'retail trade'},
    'L68': {'name': 'real estate'},
    'F42': {'name': 'construction'},
}
DEFAULT_INDUSTRY = {'name': "the firm's sector"}

def get_industry_name(sic_code):
    return INDUSTRY_CONTEXT.get(sic_code, DEFAULT_INDUSTRY)['name']

# ============================================================================
# 3. Data Loading
# ============================================================================
df_input = pd.read_csv(DATA_DIR / 'agent1_interpretation_results_4industry.csv')
print(f"Data loaded: {len(df_input)} records")

# ============================================================================
# 4. Report Generator Class (Logic-Reinforced, ratio-aware, industry-persona)
# ============================================================================
class ReportGenerator:
    def __init__(self):
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.model = os.getenv('LLM_MODEL', 'gpt-4o-mini')

    def _format_financial_guide(self, row):
        """
        Generates the numerical change guide with directional interpretation
        hints, using ratio-appropriate formatting (percent-of-base, not
        thousands-separated integers — the original absolute-value format
        would round every ratio value, e.g. 0.4230, down to 0).
        """
        changes_list = []
        features = [col.replace('Original_', '') for col in row.index if col.startswith('Original_')]

        for feat in features:
            try:
                original = float(row[f'Original_{feat}'])
                target = float(row[f'CF_{feat}'])
                change = float(row[f'Change_{feat}'])

                if abs(change) > 0.001:
                    pct_change = (change / abs(original)) * 100 if original != 0 else 0
                    direction = "increase" if change > 0 else "decrease"
                    feat_name = get_readable_name(feat)
                    base_feat = get_base_feature(feat)

                    # [Logic Guardrail] Directional hints, keyed on the BASE
                    # account code (category-safe, suffix-independent).
                    hint = ""
                    if base_feat in LIABILITY_LIKE_BASES:
                        if direction == "increase":
                            hint = " -> (Note: This is a strategic borrowing expansion to secure liquidity/working capital. Do NOT recommend 'debt repayment'.)"
                        else:
                            hint = " -> (Debt repayment to strengthen financial soundness)"
                    elif base_feat in ASSET_LIKE_BASES:
                        if direction == "decrease":
                            hint = " -> (Disposal of idle assets or inventory liquidation)"
                        else:
                            hint = " -> (Asset expansion to increase firm scale)"
                    elif base_feat in RECEIVABLE_BASES:
                        if direction == "decrease":
                            hint = " -> (Active AR collection to convert to cash)"

                    # Ratio-appropriate formatting: 4 decimal places, no
                    # thousands separator (values are proportions, typically
                    # 0-1 or small growth rates, not absolute currency amounts)
                    desc = f"- **{feat_name}**: {original:.4f} -> {target:.4f} ({abs(pct_change):.1f}% {direction}){hint}"
                    changes_list.append((abs(pct_change), desc))
            except Exception:
                continue

        changes_list.sort(key=lambda x: x[0], reverse=True)
        return "\n".join([item[1] for item in changes_list[:10]])

    def generate_report(self, row):
        company_id = row['ID']
        sic_code = row.get('SIC_CD_3', None)
        industry_name = get_industry_name(sic_code)
        financial_guide_text = self._format_financial_guide(row)
        prob_improvement = (row['Original_Proba'] - row['Target_Proba']) * 100

        system_prompt = f"""
        You are a corporate credit insurance underwriting specialist with 20 years
        of experience, specializing in the {industry_name} sector.
        Interpret the direction of numerical increases and decreases accurately
        and produce a logically consistent financial improvement report.

        [Absolute Rules]
        1. **Data-driven interpretation:** If the data indicates a 'liability increase',
           interpret it as 'securing funding' — never write 'debt repayment'.
        2. **Terminology:** Use standard financial terms instead of variable codes (FN...).
        3. **Verbatim figures:** Every CF target value from the guide MUST appear
           in the report body EXACTLY as given (same decimal precision as shown
           in the guide). Do NOT round further, approximate, or paraphrase any number.
        4. **Percentage changes:** When stating a percentage change, use the value
           already computed in the guide — do not recompute or restate it differently.
        5. **No invented numbers:** Never introduce any figure that does not appear
           in the input data. If a value is unknown, omit it rather than estimate.
        6. **Sector grounding:** Frame the diagnosis and roadmap in terms relevant
           to the {industry_name} sector's typical financial structure.
        """

        user_prompt = f"""
        [Company Information]
        - ID: {company_id}
        - Sector: {industry_name}
        - Bankruptcy probability improvement target: {row['Original_Proba']:.1%} -> {row['Target_Proba']:.1%} ({prob_improvement:.1f}pp improvement)

        [AI Analysis Summary]
        - Strategy: {row.get('selection_reason', '-')}
        - Feasibility: {row.get('feasibility_assessment', '-')}

        [Financial Indicator Change Guide (ratios, and directional hints)]
        Note: values below are proportions (e.g. 0.42 = 42% of total assets or
        revenue) or growth rates, not absolute currency amounts.
        {financial_guide_text}

        [Report Structure (Markdown)]
        # 1. Executive Summary (3-sentence overview)
        # 2. Current Status Analysis (diagnosis)
        # 3. Improvement Scenarios (detailed targets -- cite figures)
        # 4. Action Roadmap (short-term / medium-term)
        # 5. Risk Management
        # 6. Performance KPIs (3 indicators)
        """

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.5
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error: {str(e)}"

# ============================================================================
# 5. Full Execution
# ============================================================================
generator = ReportGenerator()
generated_reports = []

print(f"\n[FULL MODE] Generating reports for all {len(df_input)} firms.")
print("Running... (estimated time depends on API latency)")

from tqdm import tqdm
for idx, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Generating Reports"):
    report_text = generator.generate_report(row)
    generated_reports.append({
        'company_id': row['ID'],
        'SIC_CD_3': row.get('SIC_CD_3', None),
        'original_proba': row['Original_Proba'],
        'target_proba': row['Target_Proba'],
        'report_content': report_text
    })

# ============================================================================
# 6. Save Outputs
# ============================================================================
output_path = DATA_DIR / 'agent2_consulting_reports_4industry.csv'
df_reports = pd.DataFrame(generated_reports)
df_reports.to_csv(output_path, index=False, encoding='utf-8-sig')

print("\n" + "=" * 80)
print(f"Generation complete: {len(df_reports)} reports saved")
print(f"Combined output file: {output_path.relative_to(PROJECT_ROOT)}")
print("=" * 80)

# Save individual Markdown files (results/reports/)
for _, row in df_reports.iterrows():
    report_file = REPORTS_DIR / f"Report_{int(row['company_id'])}.md"
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write(row['report_content'])

print(f"Individual reports saved in: {REPORTS_DIR.relative_to(PROJECT_ROOT)}")

# Sample check: one report per industry
print("\n[Sample report excerpts by industry]")
for sic in ['G46', 'G47', 'L68', 'F42']:
    sector_rows = df_reports[df_reports['SIC_CD_3'] == sic]
    if not sector_rows.empty:
        sample = sector_rows.iloc[0]
        print(f"\n--- {sic} (ID: {sample['company_id']}) ---")
        print(sample['report_content'][:400])
        print("...")

Project paths ready
Step 6: Agent #2 — Consulting Report Generator (Full Run)
Data loaded: 542 records

[FULL MODE] Generating reports for all 542 firms.
Running... (estimated time depends on API latency)


Generating Reports: 100%|██████████████████████████████████████████████████████████| 542/542 [1:36:40<00:00, 10.70s/it]



Generation complete: 542 reports saved
Combined output file: data\agent2_consulting_reports_4industry.csv
Individual reports saved in: results\reports

[Sample report excerpts by industry]

--- G46 (ID: 75) ---
# 1. Executive Summary
The wholesale trade sector company has made significant strides towards financial stability, with a targeted improvement in bankruptcy probability from 81.4% to 25.9%, representing a 55.5pp improvement. Strategic expansions in borrowings and non-current liabilities indicate a proactive approach to securing liquidity and working capital. Despite the challenges indicated by th
...

--- G47 (ID: 70) ---
# 1. Executive Summary
The financial assessment of the retail trade company indicates a significant improvement in its bankruptcy probability, decreasing from 89.4% to 29.4%, representing a 60.1pp enhancement. The analysis reveals a stable foundation with a quality score of 0.89 and a minor adjustment needed to meet target financials. Key financial metrics sh